In [1]:
import requests
import os
import time
import zipfile
import pandas as pd
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types

In [2]:
client = Minio(
    "minio:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False
)
bucket = "crypto-data-lake"
if not client.bucket_exists(bucket):
    client.make_bucket(bucket)

In [3]:
spark = SparkSession.builder \
    .appName("LandingZone") \
    .config("spark.master", "spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.jars", ",".join([
        "/opt/spark-extra-jars/hadoop-aws-3.3.4.jar",
        "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.262.jar"
    ])) \
    .getOrCreate()

25/09/28 11:57:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
schema = types.StructType([
    types.StructField('agg_trade_id', types.LongType(), True), 
    types.StructField('price', types.DoubleType(), True), 
    types.StructField('quantity', types.DoubleType(), True), 
    types.StructField('first_trade_id', types.LongType(), True), 
    types.StructField('last_trade_id', types.LongType(), True), 
    types.StructField('timestamp', types.LongType(), True), 
    types.StructField('is_buyer_maker', types.BooleanType(), True), 
    types.StructField('is_best_match', types.BooleanType(), True)
])

In [5]:
def download_file(url, file_name):
    if os.path.exists(file_name):
        print(f"{file_name} exits")
        return
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(file_name, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
        print(f"Downloaded {file_name} {(os.path.getsize(file_name) / (1024 * 1024)):.2f}MB completed")

In [6]:
def extract_file(extract_dir, zip_path):
    if not os.path.exists(extract_dir):
        os.makedirs(extract_dir)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)

In [7]:
urls_month = [f"https://data.binance.vision/data/spot/monthly/aggTrades/BTCUSDT/BTCUSDT-aggTrades-2025-{i:02d}.zip" for i in range(8, 9)]

In [8]:
file_names = [u.split("/")[-1] for u in urls_month]

In [9]:
for url, file_name in zip(urls_month, file_names):
    start_t = time.time()
    download_file(url, file_name)
    # remove_file(file_name)
    end_t = time.time()
    print(f"Processed in {(end_t - start_t):.3f} seconds")

Downloaded BTCUSDT-aggTrades-2025-08.zip 350.38MB completed
Processed in 36.153 seconds


In [10]:
extract_dir = "unzipped_data_month"
zip_path = "BTCUSDT-aggTrades-2025-08.zip"
extract_file(extract_dir, zip_path)
csv_file = os.path.join(extract_dir, os.listdir(extract_dir)[0])
print(f"Extracted CSV: {csv_file}")

Extracted CSV: unzipped_data_month/BTCUSDT-aggTrades-2025-08.csv


In [11]:
df = spark.read \
    .option("header", "false") \
    .schema(schema) \
    .csv(csv_file)

In [12]:
df.printSchema()

root
 |-- agg_trade_id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: double (nullable = true)
 |-- first_trade_id: long (nullable = true)
 |-- last_trade_id: long (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- is_buyer_maker: boolean (nullable = true)
 |-- is_best_match: boolean (nullable = true)



In [13]:
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+------------+-----+--------+--------------+-------------+---------+--------------+-------------+
|agg_trade_id|price|quantity|first_trade_id|last_trade_id|timestamp|is_buyer_maker|is_best_match|
+------------+-----+--------+--------------+-------------+---------+--------------+-------------+
|           0|    0|       0|             0|            0|        0|             0|            0|
+------------+-----+--------+--------------+-------------+---------+--------------+-------------+



In [14]:
df.describe().show()

25/09/28 12:04:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 3:=====================================================>   (15 + 1) / 16]

+-------+--------------------+------------------+--------------------+--------------------+--------------------+--------------------+
|summary|        agg_trade_id|             price|            quantity|      first_trade_id|       last_trade_id|           timestamp|
+-------+--------------------+------------------+--------------------+--------------------+--------------------+--------------------+
|  count|            24302731|          24302731|            24302731|            24302731|            24302731|            24302731|
|   mean|3.6526454860000005E9|114973.14399899189|0.019395636767508686| 5.164163169957359E9| 5.164163172348015E9|1.755371134975456...|
| stddev|   7015594.286784101| 3557.921115746778| 0.14051529930130396|2.4151718262882285E7|2.4151718265063167E7|7.627175112669326E11|
|    min|          3640494121|          107350.1|              1.0E-5|          5122977554|          5122977554|    1754006400328945|
|    max|          3664796851|          124474.0|            3

In [15]:
df.sample(0.001).show(10)

+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+
|agg_trade_id|    price|quantity|first_trade_id|last_trade_id|       timestamp|is_buyer_maker|is_best_match|
+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+
|  3640494411|115730.51|   0.011|    5122978190|   5122978190|1754006417337967|          true|         true|
|  3640495717|115678.76|  2.7E-4|    5122980574|   5122980577|1754006522522261|         false|         true|
|  3640496253| 115740.0|  7.0E-5|    5122981725|   5122981726|1754006564486086|          true|         true|
|  3640496900| 115776.0|   0.002|    5122983092|   5122983092|1754006620136438|         false|         true|
|  3640496985|115784.63| 0.28037|    5122983326|   5122983342|1754006624393745|          true|         true|
|  3640499478|115740.69| 0.24663|    5122989118|   5122989119|1754006823896629|          true|         true|
|  3640499482| 1157

In [16]:
df.withColumn("ingest_date", F.current_date()).withColumn("ingest_timestamp", F.current_timestamp()).show(truncate=False)

+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-----------+--------------------------+
|agg_trade_id|price    |quantity|first_trade_id|last_trade_id|timestamp       |is_buyer_maker|is_best_match|ingest_date|ingest_timestamp          |
+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-----------+--------------------------+
|3640494121  |115764.07|0.22677 |5122977554    |5122977554   |1754006400328945|true          |true         |2025-09-28 |2025-09-28 12:05:16.336521|
|3640494122  |115764.08|0.00145 |5122977555    |5122977555   |1754006400345714|false         |true         |2025-09-28 |2025-09-28 12:05:16.336521|
|3640494123  |115764.08|2.1E-4  |5122977556    |5122977556   |1754006400350235|false         |true         |2025-09-28 |2025-09-28 12:05:16.336521|
|3640494124  |115764.08|4.1E-4  |5122977557    |5122977557   |1754006400492405|false         |true         |2025

In [17]:
df = df.withColumn("ingest_date", F.current_date()) \
    .withColumn("ingest_timestamp", F.current_timestamp())

In [18]:
output_path = f"s3a://{bucket}/landing_zone/spot/daily/aggTrades/BTCUSDT/2025_08"
df.write.mode("overwrite").parquet(output_path)
print(f"Parquet written to: {output_path}")

25/09/28 12:05:45 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Parquet written to: s3a://crypto-data-lake/landing_zone/spot/daily/aggTrades/BTCUSDT/2025_08


In [19]:
df = spark.read.parquet(output_path)

In [20]:
df.printSchema()

root
 |-- agg_trade_id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: double (nullable = true)
 |-- first_trade_id: long (nullable = true)
 |-- last_trade_id: long (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- is_buyer_maker: boolean (nullable = true)
 |-- is_best_match: boolean (nullable = true)
 |-- ingest_date: date (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)



In [21]:
df.describe().show()

[Stage 10:==============================================>           (4 + 1) / 5]

+-------+--------------------+------------------+--------------------+-------------------+--------------------+--------------------+
|summary|        agg_trade_id|             price|            quantity|     first_trade_id|       last_trade_id|           timestamp|
+-------+--------------------+------------------+--------------------+-------------------+--------------------+--------------------+
|  count|            24302731|          24302731|            24302731|           24302731|            24302731|            24302731|
|   mean|3.6526454857537713E9|114973.14399902127|0.019395636767825148|5.164163169793181E9|  5.16416317218389E9|1.755371134975247...|
| stddev|   7015594.286756441|3557.9211157456043| 0.14051529930130208|2.415171826286771E7|2.4151718265197396E7| 7.62717511288228E11|
|    min|          3640494121|          107350.1|              1.0E-5|         5122977554|          5122977554|    1754006400328945|
|    max|          3664796851|          124474.0|            39.41462

In [22]:
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

[Stage 13:==============================================>           (4 + 1) / 5]

+------------+-----+--------+--------------+-------------+---------+--------------+-------------+-----------+----------------+
|agg_trade_id|price|quantity|first_trade_id|last_trade_id|timestamp|is_buyer_maker|is_best_match|ingest_date|ingest_timestamp|
+------------+-----+--------+--------------+-------------+---------+--------------+-------------+-----------+----------------+
|           0|    0|       0|             0|            0|        0|             0|            0|          0|               0|
+------------+-----+--------+--------------+-------------+---------+--------------+-------------+-----------+----------------+



In [23]:
df.sample(0.001).show(truncate=False)

+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-----------+--------------------------+
|agg_trade_id|price    |quantity|first_trade_id|last_trade_id|timestamp       |is_buyer_maker|is_best_match|ingest_date|ingest_timestamp          |
+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-----------+--------------------------+
|3657366434  |117045.73|1.1E-4  |5180693118    |5180693118   |1755894676216611|true          |true         |2025-09-28 |2025-09-28 12:05:46.048884|
|3657368093  |116927.69|0.0023  |5180699275    |5180699280   |1755894973215487|true          |true         |2025-09-28 |2025-09-28 12:05:46.048884|
|3657369616  |116940.0 |0.01723 |5180703502    |5180703539   |1755895277458750|true          |true         |2025-09-28 |2025-09-28 12:05:46.048884|
|3657370727  |116929.96|1.0E-4  |5180707245    |5180707246   |1755895449416050|true          |true         |2025